# Experiment 1: controlled binary evidence acquisition

Comparison of Nominal, Static, HCR and Rectangular formulations on controlled finite-model examples. Outputs include values, value-of-evidence curves, answer/query/abstain regions, certificate bounds and posterior-dilation checks. No external input data are required.


## Environment

Record the active Python and numerical-library versions.


In [ ]:
import sys
import numpy as np
import pandas as pd
import scipy
import matplotlib

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Matplotlib:", matplotlib.__version__)

## Model and solvers


In [ ]:
from __future__ import annotations

import itertools
from dataclasses import dataclass
from functools import lru_cache
from typing import Dict, Iterable, List, Sequence, Tuple

import numpy as np
from scipy.optimize import linprog


Array = np.ndarray


@dataclass(frozen=True)
class SourceSpec:
    """Nominal binary source specification."""
    alpha: float  # P(E=1 | Y=1)
    beta: float   # P(E=1 | Y=0)
    cost: float
    name: str


def make_channel(alpha: float, beta: float) -> Array:
    """
    Return a binary evidence channel with rows Y={0,1} and columns E={0,1}.
    """
    if not (0.0 <= alpha <= 1.0 and 0.0 <= beta <= 1.0):
        raise ValueError("alpha and beta must lie in [0, 1].")
    return np.array(
        [
            [1.0 - beta, beta],
            [1.0 - alpha, alpha],
        ],
        dtype=float,
    )


def _deduplicate_channels(channels: Array, names: Sequence[str]) -> Tuple[Array, List[str]]:
    seen = set()
    kept_channels: List[Array] = []
    kept_names: List[str] = []

    for channel, name in zip(channels, names):
        key = tuple(np.round(channel.ravel(), 12))
        if key not in seen:
            seen.add(key)
            kept_channels.append(channel)
            kept_names.append(name)

    return np.stack(kept_channels), kept_names


def build_coupled_scenarios(
    sources: Sequence[SourceSpec],
    rho: float,
    eps: float = 1e-9,
) -> Tuple[Array, List[List[Tuple[float, float]]], Array, List[str]]:
    """
    Build a finite, coupled ambiguity set.

    The latent regimes shift all sources coherently. This deliberately creates
    dependence across source models so that the history-consistent and fully
    rectangular formulations need not coincide.

    Returns
    -------
    scenario_channels:
        Shape (n_scenarios, n_sources, 2, 2).
    source_projection_points:
        Per-source projected (alpha, beta) channel points for Rect-ROSE.
    nominal_channels:
        Shape (n_sources, 2, 2).
    scenario_names:
        Names of retained coupled regimes.
    """
    if rho < 0:
        raise ValueError("rho must be nonnegative.")

    regimes = [
        (0.0, 0.0, "nominal"),
        (-rho, +rho, "pessimistic"),
        (+rho, -rho, "optimistic"),
        (-rho, -rho, "low-positive-rate"),
        (+rho, +rho, "high-positive-rate"),
    ]

    raw_scenarios: List[Array] = []
    names: List[str] = []

    for delta_alpha, delta_beta, regime_name in regimes:
        scenario = []
        for source in sources:
            alpha = float(np.clip(source.alpha + delta_alpha, eps, 1.0 - eps))
            beta = float(np.clip(source.beta + delta_beta, eps, 1.0 - eps))
            scenario.append(make_channel(alpha, beta))
        raw_scenarios.append(np.stack(scenario))
        names.append(regime_name)

    scenario_channels, names = _deduplicate_channels(np.stack(raw_scenarios), names)

    source_projection_points: List[List[Tuple[float, float]]] = []
    for source_index in range(len(sources)):
        points: List[Tuple[float, float]] = []
        for scenario_index in range(scenario_channels.shape[0]):
            channel = scenario_channels[scenario_index, source_index]
            point = (float(channel[1, 1]), float(channel[0, 1]))
            if not any(np.allclose(point, existing) for existing in points):
                points.append(point)
        source_projection_points.append(points)

    nominal_channels = np.stack(
        [make_channel(source.alpha, source.beta) for source in sources]
    )

    return scenario_channels, source_projection_points, nominal_channels, names


def stop_value_precise(p_y1: float, abstention_cost: float) -> Tuple[float, str]:
    risks = [p_y1, 1.0 - p_y1, abstention_cost]
    actions = ["answer_0", "answer_1", "abstain"]
    index = int(np.argmin(risks))
    return float(risks[index]), actions[index]


def nominal_dp(
    prior_y1: float,
    channels: Array,
    costs: Array,
    abstention_cost: float,
) -> Tuple[Tuple[float, str, Dict[int, float]], Dict]:
    """Exact nominal dynamic program for binary sources."""

    n_sources = len(costs)
    memo: Dict = {}

    def recurse(omega: Array, remaining: Tuple[int, ...]):
        key = (tuple(float(value) for value in omega), remaining)
        if key in memo:
            return memo[key]

        total = float(omega.sum())
        p_y1 = float(omega[1] / total)
        stopping_value, stopping_action = stop_value_precise(
            p_y1, abstention_cost
        )

        best_value = stopping_value
        best_action = stopping_action
        continuation_values: Dict[int, float] = {}

        for source_index in remaining:
            value = float(costs[source_index])
            next_remaining = tuple(
                index for index in remaining if index != source_index
            )

            for evidence in (0, 1):
                next_omega = omega * channels[source_index, :, evidence]
                evidence_probability = float(next_omega.sum() / total)

                if evidence_probability > 0.0:
                    child_value = recurse(next_omega, next_remaining)[0]
                    value += evidence_probability * child_value

            continuation_values[source_index] = value

            if value < best_value - 1e-12:
                best_value = value
                best_action = f"query_{source_index}"

        memo[key] = (best_value, best_action, continuation_values)
        return memo[key]

    root = recurse(
        np.array([1.0 - prior_y1, prior_y1], dtype=float),
        tuple(range(n_sources)),
    )
    return root, memo


def nominal_dp_from_state(
    omega: Array,
    channels: Array,
    costs: Array,
    abstention_cost: float,
    remaining: Tuple[int, ...],
) -> float:
    """Nominal continuation value from an unnormalized binary belief."""

    memo: Dict = {}

    def recurse(current_omega: Array, current_remaining: Tuple[int, ...]) -> float:
        key = (tuple(float(value) for value in current_omega), current_remaining)
        if key in memo:
            return memo[key]

        total = float(current_omega.sum())
        p_y1 = float(current_omega[1] / total)
        best_value, _ = stop_value_precise(p_y1, abstention_cost)

        for source_index in current_remaining:
            value = float(costs[source_index])
            next_remaining = tuple(
                index
                for index in current_remaining
                if index != source_index
            )

            for evidence in (0, 1):
                next_omega = (
                    current_omega * channels[source_index, :, evidence]
                )
                evidence_probability = float(next_omega.sum() / total)

                if evidence_probability > 0.0:
                    value += evidence_probability * recurse(
                        next_omega, next_remaining
                    )

            best_value = min(best_value, value)

        memo[key] = float(best_value)
        return memo[key]

    return recurse(np.asarray(omega, dtype=float), remaining)


def hcr_dp(
    prior_y1: float,
    scenario_channels: Array,
    costs: Array,
    abstention_cost: float,
) -> Tuple[Tuple[float, str, Dict[int, float], Tuple[float, float, float]], Dict]:
    """
    History-Consistent Rectangular dynamic program over a finite scenario set.
    """

    n_scenarios, n_sources, _, _ = scenario_channels.shape
    memo: Dict = {}

    def likelihood_matrix(history: Tuple[Tuple[int, int], ...]) -> Array:
        likelihood = np.tile(
            np.array([1.0 - prior_y1, prior_y1], dtype=float),
            (n_scenarios, 1),
        )

        for source_index, evidence in history:
            likelihood *= scenario_channels[:, source_index, :, evidence]

        return likelihood

    def recurse(
        history: Tuple[Tuple[int, int], ...],
        remaining: Tuple[int, ...],
    ):
        key = (history, remaining)
        if key in memo:
            return memo[key]

        likelihood = likelihood_matrix(history)
        reach_probability = likelihood.sum(axis=1)
        active = reach_probability > 0.0

        if not np.any(active):
            # Every complete model assigns this history zero probability.
            memo[key] = (0.0, "abstain", {}, (0.0, 0.0, float(abstention_cost)))
            return memo[key]

        posterior_y1 = likelihood[active, 1] / reach_probability[active]

        answer_0_risk = float(np.max(posterior_y1))
        answer_1_risk = float(np.max(1.0 - posterior_y1))
        risks = [answer_0_risk, answer_1_risk, abstention_cost]
        actions = ["answer_0", "answer_1", "abstain"]

        stopping_value = float(min(risks))
        stopping_action = actions[int(np.argmin(risks))]

        best_value = stopping_value
        best_action = stopping_action
        continuation_values: Dict[int, float] = {}

        for source_index in remaining:
            next_remaining = tuple(
                index for index in remaining if index != source_index
            )

            child_values = {
                evidence: recurse(
                    history + ((source_index, evidence),),
                    next_remaining,
                )[0]
                for evidence in (0, 1)
            }

            scenario_expectations = np.zeros(n_scenarios, dtype=float)

            for evidence in (0, 1):
                numerator = (
                    likelihood
                    * scenario_channels[:, source_index, :, evidence]
                ).sum(axis=1)

                evidence_probability = np.divide(
                    numerator,
                    reach_probability,
                    out=np.zeros_like(numerator),
                    where=active,
                )

                scenario_expectations += (
                    evidence_probability * child_values[evidence]
                )

            continuation_value = float(
                costs[source_index]
                + np.max(scenario_expectations[active])
            )

            continuation_values[source_index] = continuation_value

            if continuation_value < best_value - 1e-12:
                best_value = continuation_value
                best_action = f"query_{source_index}"

        memo[key] = (
            best_value,
            best_action,
            continuation_values,
            (answer_0_risk, answer_1_risk, abstention_cost),
        )
        return memo[key]

    root = recurse(tuple(), tuple(range(n_sources)))
    return root, memo


def _deduplicate_beliefs(values: Iterable[float]) -> Tuple[float, ...]:
    return tuple(sorted(set(round(float(value), 12) for value in values)))


def rectangular_dp(
    prior_y1: float,
    source_projection_points: Sequence[Sequence[Tuple[float, float]]],
    costs: Array,
    abstention_cost: float,
) -> Tuple[Tuple[float, str, Dict[int, float], Tuple[float, float, float]], Dict]:
    """
    Fully rectangular outer approximation.

    The state retains the projected belief set but discards dependence between
    the belief and the source model that generated it.
    """

    n_sources = len(costs)
    channel_sets = [
        [make_channel(alpha, beta) for alpha, beta in source_points]
        for source_points in source_projection_points
    ]

    memo: Dict = {}

    def recurse(belief_set: Tuple[float, ...], remaining: Tuple[int, ...]):
        belief_set = _deduplicate_beliefs(belief_set)
        key = (belief_set, remaining)

        if key in memo:
            return memo[key]

        beliefs = np.asarray(belief_set, dtype=float)
        if not len(beliefs):
            memo[key] = (0.0, "abstain", {}, (0.0, 0.0, float(abstention_cost)))
            return memo[key]

        answer_0_risk = float(np.max(beliefs))
        answer_1_risk = float(np.max(1.0 - beliefs))
        risks = [answer_0_risk, answer_1_risk, abstention_cost]
        actions = ["answer_0", "answer_1", "abstain"]

        stopping_value = float(min(risks))
        stopping_action = actions[int(np.argmin(risks))]

        best_value = stopping_value
        best_action = stopping_action
        continuation_values: Dict[int, float] = {}

        for source_index in remaining:
            next_remaining = tuple(
                index for index in remaining if index != source_index
            )

            child_belief_sets: Dict[int, Tuple[float, ...]] = {}

            for evidence in (0, 1):
                posteriors: List[float] = []

                for belief in beliefs:
                    for channel in channel_sets[source_index]:
                        denominator = (
                            (1.0 - belief) * channel[0, evidence]
                            + belief * channel[1, evidence]
                        )

                        if denominator > 0.0:
                            posterior = (
                                belief * channel[1, evidence] / denominator
                            )
                            posteriors.append(float(posterior))

                child_belief_sets[evidence] = _deduplicate_beliefs(posteriors)

            child_values = {
                evidence: recurse(
                    child_belief_sets[evidence], next_remaining
                )[0]
                for evidence in (0, 1)
            }

            worst_expectation = -np.inf

            for belief in beliefs:
                for channel in channel_sets[source_index]:
                    expectation = 0.0

                    for evidence in (0, 1):
                        evidence_probability = (
                            (1.0 - belief) * channel[0, evidence]
                            + belief * channel[1, evidence]
                        )
                        expectation += (
                            evidence_probability * child_values[evidence]
                        )

                    worst_expectation = max(
                        worst_expectation, expectation
                    )

            continuation_value = float(
                costs[source_index] + worst_expectation
            )
            continuation_values[source_index] = continuation_value

            if continuation_value < best_value - 1e-12:
                best_value = continuation_value
                best_action = f"query_{source_index}"

        memo[key] = (
            best_value,
            best_action,
            continuation_values,
            (answer_0_risk, answer_1_risk, abstention_cost),
        )
        return memo[key]

    root = recurse((prior_y1,), tuple(range(n_sources)))
    return root, memo


def static_best_response(
    prior_y1: float,
    scenario_channels: Array,
    costs: Array,
    abstention_cost: float,
    adversary_distribution: Array,
):
    """
    Best deterministic policy against a distribution over fixed source models.

    The recursion returns a per-scenario risk vector. At the root, this vector
    is the expected total loss of the policy under each fixed scenario.
    """

    n_scenarios, n_sources, _, _ = scenario_channels.shape
    memo: Dict = {}

    def likelihood_matrix(history: Tuple[Tuple[int, int], ...]) -> Array:
        likelihood = np.tile(
            np.array([1.0 - prior_y1, prior_y1], dtype=float),
            (n_scenarios, 1),
        )

        for source_index, evidence in history:
            likelihood *= scenario_channels[:, source_index, :, evidence]

        return likelihood

    def recurse(
        history: Tuple[Tuple[int, int], ...],
        remaining: Tuple[int, ...],
    ):
        key = (history, remaining)

        if key in memo:
            return memo[key]

        likelihood = likelihood_matrix(history)
        reach_probability = likelihood.sum(axis=1)

        candidates = [
            ("answer_0", likelihood[:, 1].copy(), None),
            ("answer_1", likelihood[:, 0].copy(), None),
            (
                "abstain",
                abstention_cost * reach_probability.copy(),
                None,
            ),
        ]

        for source_index in remaining:
            next_remaining = tuple(
                index for index in remaining if index != source_index
            )
            child_zero = recurse(
                history + ((source_index, 0),), next_remaining
            )
            child_one = recurse(
                history + ((source_index, 1),), next_remaining
            )

            risk_vector = (
                costs[source_index] * reach_probability
                + child_zero[1]
                + child_one[1]
            )

            candidates.append(
                (
                    f"query_{source_index}",
                    risk_vector,
                    (child_zero[2], child_one[2]),
                )
            )

        weighted_values = [
            float(np.dot(adversary_distribution, candidate[1]))
            for candidate in candidates
        ]

        selected_index = int(np.argmin(weighted_values))
        action, risk_vector, children = candidates[selected_index]
        policy_tree = (action, children)

        memo[key] = (
            weighted_values[selected_index],
            risk_vector,
            policy_tree,
        )
        return memo[key]

    root = recurse(tuple(), tuple(range(n_sources)))
    return root, memo


def _solve_restricted_adversary(risk_matrix: Array):
    """
    Solve max_mu min_p <risk_p, mu> for the current policy columns.
    """

    n_policies, n_scenarios = risk_matrix.shape

    # Variables: mu_1,...,mu_S,v. Minimize -v.
    objective = np.zeros(n_scenarios + 1)
    objective[-1] = -1.0

    inequality_matrix = np.zeros((n_policies, n_scenarios + 1))
    inequality_matrix[:, :n_scenarios] = -risk_matrix
    inequality_matrix[:, -1] = 1.0
    inequality_rhs = np.zeros(n_policies)

    equality_matrix = np.zeros((1, n_scenarios + 1))
    equality_matrix[0, :n_scenarios] = 1.0
    equality_rhs = np.array([1.0])

    bounds = [(0.0, None)] * n_scenarios + [(None, None)]

    result = linprog(
        objective,
        A_ub=inequality_matrix,
        b_ub=inequality_rhs,
        A_eq=equality_matrix,
        b_eq=equality_rhs,
        bounds=bounds,
        method="highs",
    )

    if not result.success:
        raise RuntimeError(
            f"Restricted static game failed: {result.message}"
        )

    adversary_distribution = result.x[:n_scenarios]
    game_value = float(result.x[-1])

    return game_value, adversary_distribution


def static_double_oracle(
    prior_y1: float,
    scenario_channels: Array,
    costs: Array,
    abstention_cost: float,
    tolerance: float = 1e-9,
    max_iterations: int = 500,
):
    """
    Solve the exact finite-scenario randomized Static-ROSE game.

    A double-oracle method avoids enumerating all deterministic policy trees.
    The minimizer's best response is obtained by dynamic programming.
    """

    n_scenarios = scenario_channels.shape[0]
    adversary_distribution = np.ones(n_scenarios) / n_scenarios

    first_response, _ = static_best_response(
        prior_y1,
        scenario_channels,
        costs,
        abstention_cost,
        adversary_distribution,
    )

    policy_risk_vectors = [first_response[1]]
    policy_trees = [first_response[2]]
    observed_vectors = {tuple(np.round(first_response[1], 12))}
    convergence_history = []

    for iteration in range(max_iterations):
        risk_matrix = np.vstack(policy_risk_vectors)

        upper_bound, adversary_distribution = (
            _solve_restricted_adversary(risk_matrix)
        )

        best_response, _ = static_best_response(
            prior_y1,
            scenario_channels,
            costs,
            abstention_cost,
            adversary_distribution,
        )

        lower_bound = float(
            np.dot(adversary_distribution, best_response[1])
        )
        gap = float(upper_bound - lower_bound)

        convergence_history.append(
            {
                "iteration": iteration,
                "upper_bound": upper_bound,
                "lower_bound": lower_bound,
                "gap": gap,
                "n_policy_columns": len(policy_risk_vectors),
            }
        )

        if gap <= tolerance:
            return (
                upper_bound,
                adversary_distribution,
                np.vstack(policy_risk_vectors),
                policy_trees,
                convergence_history,
            )

        vector_key = tuple(np.round(best_response[1], 12))

        if vector_key in observed_vectors:
            # Numerical stalling at machine precision.
            return (
                upper_bound,
                adversary_distribution,
                np.vstack(policy_risk_vectors),
                policy_trees,
                convergence_history,
            )

        observed_vectors.add(vector_key)
        policy_risk_vectors.append(best_response[1])
        policy_trees.append(best_response[2])

    raise RuntimeError(
        "Static double-oracle solver reached its iteration limit."
    )


def _robust_stop_from_likelihood(
    likelihood: Array,
    abstention_cost: float,
) -> float:
    reach_probability = likelihood.sum(axis=1)
    active = reach_probability > 0.0

    if not np.any(active):
        return 0.0

    posterior_y1 = (
        likelihood[active, 1] / reach_probability[active]
    )

    return float(
        min(
            np.max(posterior_y1),
            np.max(1.0 - posterior_y1),
            abstention_cost,
        )
    )


def root_acquisition_certificates(
    prior_y1: float,
    scenario_channels: Array,
    costs: Array,
    abstention_cost: float,
):
    """
    Compute root-level perfect-information lower bounds and query-then-stop
    upper bounds for all source actions.
    """

    import pandas as pd

    n_scenarios, n_sources, _, _ = scenario_channels.shape
    root_likelihood = np.tile(
        np.array([1.0 - prior_y1, prior_y1], dtype=float),
        (n_scenarios, 1),
    )

    hcr_root, _ = hcr_dp(
        prior_y1, scenario_channels, costs, abstention_cost
    )
    stopping_value = float(min(hcr_root[3]))

    rows = []

    for source_index in range(n_sources):
        remaining = tuple(
            index for index in range(n_sources) if index != source_index
        )

        child_stopping_values = {}
        for evidence in (0, 1):
            next_likelihood = (
                root_likelihood
                * scenario_channels[:, source_index, :, evidence]
            )
            child_stopping_values[evidence] = (
                _robust_stop_from_likelihood(
                    next_likelihood, abstention_cost
                )
            )

        lower_scenario_values = []
        upper_scenario_values = []

        for scenario_index in range(n_scenarios):
            perfect_information_expectation = 0.0
            query_then_stop_expectation = 0.0

            for evidence in (0, 1):
                next_omega = (
                    root_likelihood[scenario_index]
                    * scenario_channels[
                        scenario_index, source_index, :, evidence
                    ]
                )
                evidence_probability = float(next_omega.sum())

                if evidence_probability > 0.0:
                    perfect_information_expectation += (
                        evidence_probability
                        * nominal_dp_from_state(
                            next_omega,
                            scenario_channels[scenario_index],
                            costs,
                            abstention_cost,
                            remaining,
                        )
                    )
                    query_then_stop_expectation += (
                        evidence_probability
                        * child_stopping_values[evidence]
                    )

            lower_scenario_values.append(
                costs[source_index] + perfect_information_expectation
            )
            upper_scenario_values.append(
                costs[source_index] + query_then_stop_expectation
            )

        lower_bound = float(max(lower_scenario_values))
        upper_bound = float(max(upper_scenario_values))
        exact_hcr_value = float(
            hcr_root[2][source_index]
        )

        rows.append(
            {
                "source_index": source_index,
                "LB": lower_bound,
                "C_HCR": exact_hcr_value,
                "UB": upper_bound,
                "stop_value": stopping_value,
                "bound_valid": (
                    lower_bound
                    <= exact_hcr_value + 1e-10
                    and exact_hcr_value
                    <= upper_bound + 1e-10
                ),
                "rejection_certificate": (
                    lower_bound >= stopping_value - 1e-12
                ),
                "no_harm_certificate": (
                    upper_bound < stopping_value - 1e-12
                ),
            }
        )

    return pd.DataFrame(rows)


def posterior_dilation_unit_test() -> Dict[str, float]:
    """
    Reproduce the analytical cost-free retrieval dilation example.
    """

    prior_y1 = 0.9
    abstention_cost = 0.4
    costs = np.array([0.0])

    channel_a = make_channel(alpha=0.9, beta=0.1)
    channel_b = make_channel(alpha=0.1, beta=0.9)

    scenarios = np.stack(
        [
            np.stack([channel_a]),
            np.stack([channel_b]),
        ]
    )

    root, _ = hcr_dp(
        prior_y1, scenarios, costs, abstention_cost
    )

    initial_stopping_value = float(min(root[3]))
    continuation_value = float(root[2][0])
    robust_value_of_evidence = (
        initial_stopping_value - continuation_value
    )

    expected = {
        "initial_stopping_value": 0.1,
        "post_retrieval_hcr_value": 0.4,
        "RVoE_HCR": -0.3,
    }

    actual = {
        "initial_stopping_value": initial_stopping_value,
        "post_retrieval_hcr_value": continuation_value,
        "RVoE_HCR": robust_value_of_evidence,
    }

    for key, expected_value in expected.items():
        if not np.isclose(actual[key], expected_value, atol=1e-10):
            raise AssertionError(
                f"Dilation unit test failed for {key}: "
                f"expected {expected_value}, obtained {actual[key]}."
            )

    return actual

## Experiment configuration and evaluation


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Dict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "rose_rag_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# Remove raster figures left by older versions of the experiment.
for old_png in OUTPUT_DIR.glob("*.png"):
    old_png.unlink()



def save_vector_figure(figure, stem: str):
    """
    Save a Matplotlib figure only in vector formats.

    Outputs:
    - PDF: preferred for Overleaf and publication
    - SVG: convenient for inspection and editing
    """
    pdf_path = OUTPUT_DIR / f"{stem}.pdf"
    svg_path = OUTPUT_DIR / f"{stem}.svg"

    figure.savefig(
        pdf_path,
        format="pdf",
        bbox_inches="tight",
    )
    figure.savefig(
        svg_path,
        format="svg",
        bbox_inches="tight",
    )



SOURCES = [
    SourceSpec(
        alpha=0.90,
        beta=0.10,
        cost=0.08,
        name="S1_high_quality",
    ),
    SourceSpec(
        alpha=0.75,
        beta=0.25,
        cost=0.03,
        name="S2_medium_quality",
    ),
    SourceSpec(
        alpha=0.65,
        beta=0.35,
        cost=0.01,
        name="S3_low_cost",
    ),
]

PRIOR_Y1 = 0.70
ABSTENTION_COST = 0.30
RHO_GRID = np.array(
    [0.00, 0.02, 0.05, 0.08, 0.10, 0.12, 0.15, 0.18, 0.20]
)


def run_robustness_ladder():
    costs = np.array([source.cost for source in SOURCES])
    hierarchy_rows = []
    rvoe_rows = []

    for rho in RHO_GRID:
        (
            scenario_channels,
            source_projection_points,
            nominal_channels,
            scenario_names,
        ) = build_coupled_scenarios(SOURCES, float(rho))

        nominal_root, _ = nominal_dp(
            PRIOR_Y1,
            nominal_channels,
            costs,
            ABSTENTION_COST,
        )

        hcr_root, _ = hcr_dp(
            PRIOR_Y1,
            scenario_channels,
            costs,
            ABSTENTION_COST,
        )

        rectangular_root, _ = rectangular_dp(
            PRIOR_Y1,
            source_projection_points,
            costs,
            ABSTENTION_COST,
        )

        (
            static_value,
            _,
            _,
            _,
            static_history,
        ) = static_double_oracle(
            PRIOR_Y1,
            scenario_channels,
            costs,
            ABSTENTION_COST,
            tolerance=1e-10,
        )

        nominal_value = float(nominal_root[0])
        hcr_value = float(hcr_root[0])
        rectangular_value = float(rectangular_root[0])
        stopping_value = float(min(hcr_root[3]))

        tolerance = 1e-8
        hierarchy_valid = (
            nominal_value <= static_value + tolerance
            and static_value <= hcr_value + tolerance
            and hcr_value <= rectangular_value + tolerance
        )

        hierarchy_rows.append(
            {
                "rho": rho,
                "n_coupled_scenarios": len(scenario_names),
                "V_nominal": nominal_value,
                "V_static": static_value,
                "V_hcr": hcr_value,
                "V_rect": rectangular_value,
                "action_nominal": nominal_root[1],
                "action_hcr": hcr_root[1],
                "action_rect": rectangular_root[1],
                "PoA": static_value - nominal_value,
                "PoDC": hcr_value - static_value,
                "PoR": rectangular_value - hcr_value,
                "hierarchy_valid": hierarchy_valid,
                "static_double_oracle_iterations": len(static_history),
            }
        )

        for source_index, source in enumerate(SOURCES):
            continuation_value = float(
                hcr_root[2][source_index]
            )
            rvoe_rows.append(
                {
                    "rho": rho,
                    "source": source.name,
                    "RVoE_HCR": (
                        stopping_value - continuation_value
                    ),
                    "continuation_value": continuation_value,
                    "stop_value": stopping_value,
                }
            )

    hierarchy = pd.DataFrame(hierarchy_rows)
    rvoe = pd.DataFrame(rvoe_rows)

    hierarchy.to_csv(
        OUTPUT_DIR / "results_robustness_hierarchy.csv",
        index=False,
    )
    rvoe.to_csv(
        OUTPUT_DIR / "results_rvoe_by_radius.csv",
        index=False,
    )

    figure = plt.figure(figsize=(8, 5))
    plt.plot(
        hierarchy["rho"],
        hierarchy["V_nominal"],
        marker="o",
        label="Nominal",
    )
    plt.plot(
        hierarchy["rho"],
        hierarchy["V_static"],
        marker="o",
        label="Static-ROSE",
    )
    plt.plot(
        hierarchy["rho"],
        hierarchy["V_hcr"],
        marker="o",
        label="HCR-ROSE",
    )
    plt.plot(
        hierarchy["rho"],
        hierarchy["V_rect"],
        marker="o",
        label="Rect-ROSE",
    )
    plt.xlabel("Ambiguity radius")
    plt.ylabel("Optimal decision risk")
    plt.title("ROSE-RAG robustness hierarchy")
    plt.legend()
    plt.tight_layout()
    save_vector_figure(figure, "figure_robustness_hierarchy")
    plt.close(figure)

    figure = plt.figure(figsize=(8, 5))
    for source_name, group in rvoe.groupby("source"):
        plt.plot(
            group["rho"],
            group["RVoE_HCR"],
            marker="o",
            label=source_name,
        )
    plt.axhline(0.0)
    plt.xlabel("Ambiguity radius")
    plt.ylabel("HCR Robust Value of Evidence")
    plt.title("Robust Value of Evidence versus ambiguity")
    plt.legend()
    plt.tight_layout()
    save_vector_figure(figure, "figure_rvoe_vs_radius")
    plt.close(figure)

    return hierarchy, rvoe


def run_action_region():
    costs = np.array([source.cost for source in SOURCES])

    prior_grid = np.linspace(0.01, 0.99, 61)
    rho_grid = np.linspace(0.0, 0.20, 41)

    action_labels = [
        "answer_0",
        "abstain",
        "query_0",
        "query_1",
        "query_2",
        "answer_1",
    ]
    action_to_code: Dict[str, int] = {
        action: index for index, action in enumerate(action_labels)
    }

    action_matrix = np.zeros(
        (len(rho_grid), len(prior_grid)),
        dtype=int,
    )

    rows = []

    for rho_index, rho in enumerate(rho_grid):
        (
            scenario_channels,
            _,
            _,
            _,
        ) = build_coupled_scenarios(SOURCES, float(rho))

        for prior_index, prior in enumerate(prior_grid):
            root, _ = hcr_dp(
                float(prior),
                scenario_channels,
                costs,
                ABSTENTION_COST,
            )

            action = root[1]
            action_matrix[rho_index, prior_index] = (
                action_to_code[action]
            )

            rows.append(
                {
                    "prior_y1": prior,
                    "rho": rho,
                    "action": action,
                    "optimal_risk": root[0],
                }
            )

    pd.DataFrame(rows).to_csv(
        OUTPUT_DIR / "results_action_regions.csv",
        index=False,
    )

    figure = plt.figure(figsize=(9, 5.5))
    image = plt.imshow(
        action_matrix,
        origin="lower",
        aspect="auto",
        extent=[
            prior_grid.min(),
            prior_grid.max(),
            rho_grid.min(),
            rho_grid.max(),
        ],
    )
    colorbar = plt.colorbar(image)
    colorbar.set_ticks(range(len(action_labels)))
    colorbar.set_ticklabels(action_labels)
    plt.xlabel("Internal prior P(Y=1)")
    plt.ylabel("Ambiguity radius")
    plt.title("HCR-ROSE answer–retrieve–abstain regions")
    plt.tight_layout()
    save_vector_figure(figure, "figure_action_regions")
    plt.close(figure)


def run_certificates():
    costs = np.array([source.cost for source in SOURCES])
    rho = 0.10

    (
        scenario_channels,
        _,
        _,
        _,
    ) = build_coupled_scenarios(SOURCES, rho)

    certificates = root_acquisition_certificates(
        PRIOR_Y1,
        scenario_channels,
        costs,
        ABSTENTION_COST,
    )

    certificates["source"] = [
        source.name for source in SOURCES
    ]

    certificates.to_csv(
        OUTPUT_DIR / "results_root_certificates.csv",
        index=False,
    )

    figure = plt.figure(figsize=(8, 5))
    positions = np.arange(len(certificates))
    width = 0.25

    plt.bar(
        positions - width,
        certificates["LB"],
        width=width,
        label="Lower bound",
    )
    plt.bar(
        positions,
        certificates["C_HCR"],
        width=width,
        label="Exact HCR continuation",
    )
    plt.bar(
        positions + width,
        certificates["UB"],
        width=width,
        label="Upper bound",
    )
    plt.axhline(
        float(certificates["stop_value"].iloc[0]),
        label="Stop value",
    )
    plt.xticks(positions, certificates["source"], rotation=15)
    plt.ylabel("Risk")
    plt.title("Root acquisition certificates at rho=0.10")
    plt.legend()
    plt.tight_layout()
    save_vector_figure(figure, "figure_certificate_bounds")
    plt.close(figure)

    return certificates


def estimate_critical_radii(rvoe: pd.DataFrame):
    rows = []

    for source_name, group in rvoe.groupby("source"):
        group = group.sort_values("rho")
        positive = group[group["RVoE_HCR"] > 0.0]
        nonpositive = group[group["RVoE_HCR"] <= 0.0]

        lower = (
            float(positive["rho"].max())
            if not positive.empty
            else np.nan
        )
        upper = (
            float(nonpositive["rho"].min())
            if not nonpositive.empty
            else np.nan
        )

        rows.append(
            {
                "source": source_name,
                "last_positive_grid_radius": lower,
                "first_nonpositive_grid_radius": upper,
                "critical_radius_interval": (
                    f"({lower}, {upper}]"
                    if not np.isnan(lower)
                    and not np.isnan(upper)
                    else "not bracketed"
                ),
            }
        )

    result = pd.DataFrame(rows)
    result.to_csv(
        OUTPUT_DIR / "results_critical_radius_brackets.csv",
        index=False,
    )
    return result


def write_summary(
    hierarchy: pd.DataFrame,
    rvoe: pd.DataFrame,
    certificates: pd.DataFrame,
    dilation_result: Dict[str, float],
):
    critical = estimate_critical_radii(rvoe)

    summary_lines = [
        "ROSE-RAG Controlled Binary Validation",
        "=" * 42,
        "",
        "Configuration:",
        f"- Prior P(Y=1): {PRIOR_Y1}",
        f"- Abstention cost: {ABSTENTION_COST}",
        f"- Sources: {len(SOURCES)}",
        "",
        "Checks:",
        (
            "- Robustness hierarchy valid for every radius: "
            f"{bool(hierarchy['hierarchy_valid'].all())}"
        ),
        (
            "- All root certificate bounds valid at rho=0.10: "
            f"{bool(certificates['bound_valid'].all())}"
        ),
        (
            "- Posterior dilation unit test reproduced exactly: "
            f"{dilation_result}"
        ),
        "",
        "Critical-radius grid brackets:",
        critical.to_string(index=False),
        "",
        "Important interpretation:",
        (
            "The coupled scenario model preserves cross-source dependence. "
            "Rect-ROSE enlarges this set by allowing arbitrary combinations "
            "of source projections, so V_rect can exceed V_hcr."
        ),
        "",
        "This is a controlled finite-scenario validation. It is not yet a "
        "public-benchmark result and must not be described as empirical "
        "superiority over existing RAG systems.",
    ]

    (OUTPUT_DIR / "summary.txt").write_text(
        "\n".join(summary_lines),
        encoding="utf-8",
    )


def main():
    hierarchy, rvoe = run_robustness_ladder()
    run_action_region()
    certificates = run_certificates()
    dilation_result = posterior_dilation_unit_test()

    with (OUTPUT_DIR / "results_dilation_unit_test.json").open(
        "w", encoding="utf-8"
    ) as file:
        json.dump(dilation_result, file, indent=2)

    write_summary(
        hierarchy,
        rvoe,
        certificates,
        dilation_result,
    )

    print("Experiment 1 completed successfully.")
    print(f"Outputs written to: {OUTPUT_DIR}")
    print()
    print(hierarchy[
        [
            "rho",
            "V_nominal",
            "V_static",
            "V_hcr",
            "V_rect",
            "action_hcr",
            "hierarchy_valid",
        ]
    ].to_string(index=False))
    print()
    print("Posterior dilation unit test:")
    print(json.dumps(dilation_result, indent=2))



if __name__ == '__main__':
    main()

## Run the experiment

Tables and PDF/SVG figures are written under `rose_rag_outputs`.


In [ ]:
main()

## Result tables


In [ ]:
from IPython.display import display

output_dir = Path.cwd() / "rose_rag_outputs"

hierarchy_df = pd.read_csv(output_dir / "results_robustness_hierarchy.csv")
rvoe_df = pd.read_csv(output_dir / "results_rvoe_by_radius.csv")
cert_df = pd.read_csv(output_dir / "results_root_certificates.csv")

print("Robustness hierarchy:")
display(hierarchy_df)

print("\nRoot certificates:")
display(cert_df)

print("\nRVoE by ambiguity radius:")
display(rvoe_df.head(20))

## Saved figures


In [ ]:
from IPython.display import SVG, display

for filename in [
    "figure_robustness_hierarchy.svg",
    "figure_rvoe_vs_radius.svg",
    "figure_action_regions.svg",
    "figure_certificate_bounds.svg",
]:
    print(filename)
    display(SVG(filename=str(output_dir / filename)))